# SFT — Qwen3-0.6B on Synthetic CoT Data

**Step 1 of the hybrid approach.** Fine-tune Qwen3-0.6B on the synthetic QA dataset
generated by DeepSeek-V4-Flash. Each training example has a `<think>` reasoning block
followed by the final answer — the model learns both *how to reason* about the manual
and *what the correct facts are* simultaneously.

After this, run `onpolicy_distill_qwen3.ipynb` (Step 2) to refine under the model's
own generation distribution.

**Runtime:** Colab → Runtime → Change runtime type → **L4 GPU**  
**Data:** Upload `ac_manual_synth_tra.jsonl` and `ac_manual_synth_val.jsonl` to Colab.


In [ ]:
# 0. Install. After this runs: Runtime → Restart session, then Run all.
!pip install -q "transformers>=4.51" "peft>=0.13" "trl>=0.12" accelerate datasets


In [ ]:
# 1. Config
import json, random
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer, SFTConfig

MODEL_ID    = "Qwen/Qwen3-0.6B"
TRAIN_JSONL = "ac_manual_synth_tra.jsonl"
VAL_JSONL   = "ac_manual_synth_val.jsonl"
OUTPUT_DIR  = "./qwen3_06b_ac_sft_lora"

EPOCHS       = 3
MAX_SEQ_LEN  = 2048    # think blocks can be long
BATCH_SIZE   = 2
ACCUM_STEPS  = 4       # effective batch = 8
LR           = 2e-4
LORA_R       = 32
SEED         = 0
MAX_NEW_TOKENS = 512

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE  = torch.bfloat16 if DEVICE == "cuda" else torch.float32
random.seed(SEED); torch.manual_seed(SEED)
print("device:", DEVICE, "| dtype:", DTYPE)


In [ ]:
# 2. Load data
def load_jsonl(path):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return Dataset.from_list(rows)

train_ds = load_jsonl(TRAIN_JSONL)
val_ds   = load_jsonl(VAL_JSONL)
print(f"Train: {len(train_ds)} examples")
print(f"Val:   {len(val_ds)} examples")
print("\nSample text (first 300 chars):")
print(train_ds[0]["text"][:300])


In [ ]:
# 3. Tokenizer + model + LoRA
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
# Left-pad for generation; right-pad for training (SFTTrainer handles this internally)
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=DTYPE).to(DEVICE)
model = get_peft_model(model, LoraConfig(
    r=LORA_R, lora_alpha=64, lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
))
model.print_trainable_parameters()


## Before training

Baseline greedy answers from the untrained model. Answers will be wrong — that's expected.
After SFT the think blocks should cite correct manual specs.


In [ ]:
SYSTEM = ("You are a helpful assistant for the Sharp CV-P09FX portable air conditioner. "
          "Answer questions accurately based on the product manual.")

@torch.no_grad()
def answer(question):
    """Greedy student answer with thinking enabled."""
    messages = [{"role": "system", "content": SYSTEM},
                {"role": "user",   "content": question}]
    ids = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, enable_thinking=True,
        return_tensors="pt", return_dict=True,
    )["input_ids"].to(DEVICE)
    out = model.generate(
        ids, attention_mask=torch.ones_like(ids),
        max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
    )
    return tokenizer.decode(out[0, ids.shape[1]:], skip_special_tokens=True).strip()

eval_questions = [
    "What is the minimum window width for the window panel?",
    "How many screws do I use for a 30-inch wide window?",
    "What should I do if my window width is between 22 and 24 inches?",
]

model.eval()
for q in eval_questions:
    print(f"Q: {q}\nA: {answer(q)}\n")


## Train

Loss is computed only on the assistant turn (system + user prompts are masked) so the
model focuses on learning to generate `<think>` reasoning chains and correct answers.
Watch `eval_loss` drop across epochs — lower = model is predicting the CoT answers better.


In [ ]:
# Manual completion-only collator — masks system+user prompt tokens so loss is
# computed only on the assistant turn (including <think> block and final answer).

# Tokenize the dataset once; keep only assistant-turn tokens in labels.
ASSISTANT_HEADER = tokenizer.encode("<|im_start|>assistant\n", add_special_tokens=False)

def tokenize_and_mask(example):
    ids = tokenizer(
        example["text"],
        truncation=True,
        max_length=MAX_SEQ_LEN,
        padding=False,
    )["input_ids"]

    labels = list(ids)

    # Mask everything before the last <|im_start|>assistant\n occurrence
    n = len(ids)
    h = len(ASSISTANT_HEADER)
    split_pos = None
    for i in range(n - h, -1, -1):
        if ids[i:i + h] == ASSISTANT_HEADER:
            split_pos = i + h
            break

    if split_pos is not None:
        for j in range(split_pos):
            labels[j] = -100

    return {"input_ids": ids, "labels": labels}

tok_train = train_ds.map(tokenize_and_mask, remove_columns=["text"])
tok_val   = val_ds.map(tokenize_and_mask,   remove_columns=["text"])

from dataclasses import dataclass

@dataclass
class CompletionCollator:
    pad_id: int
    def __call__(self, features):
        max_len = max(len(f["input_ids"]) for f in features)
        input_ids, labels, masks = [], [], []
        for f in features:
            pad = max_len - len(f["input_ids"])
            input_ids.append(f["input_ids"] + [self.pad_id] * pad)
            labels.append(f["labels"]       + [-100]       * pad)
            masks.append([1] * len(f["input_ids"]) + [0] * pad)
        return {
            "input_ids":      torch.tensor(input_ids),
            "attention_mask": torch.tensor(masks),
            "labels":         torch.tensor(labels),
        }

collator = CompletionCollator(pad_id=tokenizer.pad_token_id)

# Detect which SFTConfig parameter name this TRL version uses
import inspect
_sft_params = inspect.signature(SFTConfig.__init__).parameters
_seq_len_kwarg = "max_seq_length" if "max_seq_length" in _sft_params else "max_length"
print(f"TRL SFTConfig uses: {_seq_len_kwarg}")

trainer = SFTTrainer(
    model=model,
    train_dataset=tok_train,
    eval_dataset=tok_val,
    data_collator=collator,
    args=SFTConfig(
        output_dir=OUTPUT_DIR,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=ACCUM_STEPS,
        learning_rate=LR,
        bf16=(DTYPE == torch.bfloat16),
        logging_steps=5,
        eval_strategy="epoch",
        save_strategy="best",
        metric_for_best_model="eval_loss",
        load_best_model_at_end=True,
        dataset_text_field=None,
        **{_seq_len_kwarg: MAX_SEQ_LEN},
        seed=SEED,
        report_to="none",
        remove_unused_columns=False,
    ),
)

trainer.train()


## After training

Check whether the think blocks now reason toward correct specs (22 inches / 559mm,
correct screw counts, etc.). If yes, this model is a strong starting point for
Step 2: on-policy distillation in `onpolicy_distill_qwen3.ipynb`.


In [ ]:
model.eval()
for q in eval_questions:
    print(f"Q: {q}\nA: {answer(q)}\n")

model.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Saved adapter + tokenizer to {OUTPUT_DIR}")
print("\nNext step: load this adapter as the student in onpolicy_distill_qwen3.ipynb")
